
# Product Price Prediction. Fine-Tuning GPT-4.1-nano
A fine-tuned LLM that estimates Amazon product prices from descriptions. Play "The Price is Right": guess the price, then see how you stack up
against the base model and the fine-tuned model side by side.

In [2]:
# imports

import os
import re
import json
import random
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
import gradio as gr
import sys
sys.path.append('../../')
from pricer.items import Item
from pricer.evaluator import evaluate

In [3]:
# constants

BASE_MODEL   = "gpt-4.1-nano-2025-04-14"
TRAIN_SIZE   = 200
VAL_SIZE     = 50
JSONL_FOLDER = "jsonl"

SYSTEM_PROMPT = (
    "You are a product pricing assistant with deep knowledge of the Amazon marketplace. "
    "Given a product title, category, brand, and description, estimate its price in USD. "
    "Respond with the price only, in the format: Price is $X.XX. No explanation."
)

In [ ]:
# set up environment
load_dotenv(override=True)

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("✅ HuggingFace token found")
else:
    print("❌ HuggingFace token not found")

openai_api_key = os.environ.get("OPENAI_API_KEY")
if openai_api_key:
    print("✅ OpenAI API key found")
else:
    print("❌ OpenAI API key not found")

openai = OpenAI()


In [ ]:
# load dataset

train, val, test = Item.from_hub("ed-donner/items_lite")
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

fine_tune_train = train[:TRAIN_SIZE]
fine_tune_val   = val[:VAL_SIZE]


In [ ]:
# prepare training data

def messages_for(item):
    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Estimate the price of this product.\n\n{item.summary}"},
        {"role": "assistant", "content": f"Price is ${item.price:.2f}"},
    ]

def make_jsonl(items):
    result = ""
    for item in items:
        messages     = messages_for(item)
        messages_str = json.dumps(messages)
        result      += '{"messages": ' + messages_str + '}\n'
    return result.strip()

def write_jsonl(items, filename):
    Path(JSONL_FOLDER).mkdir(exist_ok=True)
    with open(filename, "w") as f:
        f.write(make_jsonl(items))

write_jsonl(fine_tune_train, f"{JSONL_FOLDER}/fine_tune_train.jsonl")
write_jsonl(fine_tune_val,   f"{JSONL_FOLDER}/fine_tune_validation.jsonl")
print("✅ JSONL files written")


print(json.dumps({"messages": messages_for(fine_tune_train[0])}, indent=2))

In [ ]:
# upload to OpenAI

with open(f"{JSONL_FOLDER}/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
print(f"✅ Train file uploaded: {train_file.id}")

with open(f"{JSONL_FOLDER}/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")
print(f"✅ Validation file uploaded: {validation_file.id}")

In [ ]:
# submit fine-tuning job

job = openai.fine_tuning.jobs.create(
    training_file   = train_file.id,
    validation_file = validation_file.id,
    model           = BASE_MODEL,
    seed            = 42,
    hyperparameters = {"n_epochs": 1, "batch_size": 1},
    suffix          = "pricer"
)
print(f"✅ Fine-tuning job submitted: {job.id}")
print("Track progress at: https://platform.openai.com/finetune")

In [ ]:
# Monitor Job
# Re-run this cell until status shows 'succeeded'
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
status = openai.fine_tuning.jobs.retrieve(job_id)
print(f"Job ID : {job_id}")
print(f"Status : {status.status}")

recent_events = openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=5).data
for event in recent_events:
    print(f"  {event.message}")

In [ ]:
# Evaluate base model

def test_messages_for(item):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Estimate the price of this product.\n\n{item.summary}"},
    ]

def base_model_pricer(item):
    response = openai.chat.completions.create(
        model    = BASE_MODEL,
        messages = test_messages_for(item),
        max_tokens = 7
    )
    return response.choices[0].message.content

print("Evaluating base model (no fine-tuning)...")
evaluate(base_model_pricer, test)

In [ ]:
# Evaluate Fine-Tuned Model
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
print(f"Fine-tuned model: {fine_tuned_model_name}")

def fine_tuned_pricer(item):
    response = openai.chat.completions.create(
        model      = fine_tuned_model_name,
        messages   = test_messages_for(item),
        max_tokens = 7
    )
    return response.choices[0].message.content

print("Evaluating fine-tuned model...")
evaluate(fine_tuned_pricer, test)

In [ ]:
# Gradio UI
def get_price(text):
    s     = text.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else 0.0

def load_product():
    item        = random.choice(test)
    description = item.summary
    return description, "", "", gr.update(interactive=True)

def submit_guess(description, user_guess):
    item       = next(i for i in test if i.summary == description)
    actual     = item.price
    user_price = get_price(user_guess)

    base_response = openai.chat.completions.create(
        model      = BASE_MODEL,
        messages   = test_messages_for(item),
        max_tokens = 7
    ).choices[0].message.content
    base_price = get_price(base_response)

    ft_response = openai.chat.completions.create(
        model      = fine_tuned_model_name,
        messages   = test_messages_for(item),
        max_tokens = 7
    ).choices[0].message.content
    ft_price = get_price(ft_response)

    result = (
        f"Actual price:         ${actual:.2f}\n\n"
        f"Your guess:           ${user_price:.2f}  (error: ${abs(user_price - actual):.2f})\n"
        f"Base model:           ${base_price:.2f}  (error: ${abs(base_price - actual):.2f})\n"
        f"Fine-tuned model:     ${ft_price:.2f}  (error: ${abs(ft_price - actual):.2f})"
    )
    return result

welcome = [{
    "role":    "assistant",
    "content": (
        "🛒 Welcome to the Price is Right!\n\n"
        "A product description will be shown below. Guess its price, then hit Submit "
        "to see how you compare against the base model and the fine-tuned model.\n\n"
        "Hit 'Next Product' to begin!"
    )
}]

with gr.Blocks() as ui:
    gr.Markdown("## 🏷️ The Price is Right — LLM Fine-Tuning Edition")
    gr.Markdown("Guess the product price, then see how you compare against the base and fine-tuned models.")

    chatbot     = gr.Chatbot(value=welcome, height=200, type="messages")
    description = gr.Textbox(label="Product Description", lines=8, interactive=False)
    user_guess  = gr.Textbox(label="Your Price Guess (e.g. 49.99)", placeholder="Enter your guess...")
    result_box  = gr.Textbox(label="Results", lines=6, interactive=False)

    with gr.Row():
        next_btn   = gr.Button("Next Product", variant="secondary")
        submit_btn = gr.Button("Submit Guess", variant="primary")

    next_btn.click(
        fn      = lambda: gr.update(interactive=False),
        outputs = next_btn
    ).then(
        fn      = load_product,
        outputs = [description, user_guess, result_box, submit_btn]
    ).then(
        fn      = lambda: gr.update(interactive=True),
        outputs = next_btn
    )

    submit_btn.click(
        fn      = lambda: gr.update(interactive=False),
        outputs = submit_btn
    ).then(
        fn      = submit_guess,
        inputs  = [description, user_guess],
        outputs = result_box
    ).then(
        fn      = lambda: gr.update(interactive=True),
        outputs = submit_btn
    )

ui.launch(inbrowser=True)